#### py16S: A 16S Microbial Community Analysis

This notebook will accept outputs generated from QIIME2's workflow (specific workflow: https://github.com/shelleydan/metabarcoding). This will include the following files:
- Metadata (CSV)
- Taxonomy Table (TSV)
- Feature Count Table (TSV)

It is recommended that this workbook is ran chunk by chunk. 

##### (1.) Library Import

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("py16S/src")
import biodiversity as bio

##### (0.) Assigning User Variables

In [5]:
# Import File Paths
metadata_path        = "C:/Users/Dan_PC/OneDrive - Cardiff University/004 - Cardiff University - Senior Research Technician/Add to Github/DCWW_HRUU_Qiime2_Output/metadata.csv"
otu_table_path       = "C:/Users/Dan_PC/OneDrive - Cardiff University/004 - Cardiff University - Senior Research Technician/Add to Github/DCWW_HRUU_Qiime2_Output/feature-table.tsv"
taxonomy_table_path  = "C:/Users/Dan_PC/OneDrive - Cardiff University/004 - Cardiff University - Senior Research Technician/Add to Github/DCWW_HRUU_Qiime2_Output/taxonomy.tsv"

# Set Output Directory
output_directory     = "C:/Users/Dan_PC/OneDrive - Cardiff University/004 - Cardiff University - Senior Research Technician/py16S/output/"

##### (1.) Data import and cleaning.

In [36]:
# Import Data
metadata_df = pd.read_csv(metadata_path)
otu_table_df = pd.read_csv(otu_table_path, sep='\t')
taxonomy_table_df = pd.read_csv(taxonomy_table_path, sep='\t')

# Cleaning Taxonomy Table
ranks = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]

tax_cols = (
    taxonomy_table_df["Taxon"]
    .str.split(";", expand=True)
    .iloc[:, :len(ranks)]
)

tax_cols.columns = ranks[:tax_cols.shape[1]]

tax_cols = tax_cols.apply(
    lambda col: (
        col
        .str.strip()
        .str.replace(r"^[a-zA-Z0-9_]+__", "", regex=True)
        .replace("", pd.NA)
    )
)

taxonomy_table_df = pd.concat([taxonomy_table_df, tax_cols], axis=1)
taxonomy_table_df = taxonomy_table_df.drop(columns=["Taxon"])

# Combining the Taxonomy table with the OTU table
otu_table_df = otu_table_df.reset_index().rename(columns={"index": "Feature ID"})
otu_taxonomy_df = pd.merge(
    taxonomy_table_df,
    otu_table_df,
    left_on="Feature ID",
    right_on="Feature ID",
    how="left"
)

(2.) Biodiversity Curve

In [82]:
bio.biodiversity_curve(otu_table_df, max_depth=50000, depth_step=1000)

KeyboardInterrupt: 